### Script for generating figures of broadband ranges for kmeans paper.

In [ ]:
import sys
sys.path.append("..")

In [ ]:
import numpy as np
from plotly import graph_objects as go
import scipy.ndimage as ndi

In [ ]:
from wavelet_transform.utils.save_plots import save_figure

from wavelet_transform.utils.extensions import psd, to_dB
import wavelet_transform.utils.kmeans_results as results
import wavelet_transform.utils.statistical_features as sf
import wavelet_transform.wpt.inverse_transform as inv

In [ ]:
savefigs_flag = True

In [ ]:
def trim(data, ratio):
    lim = int((1 - ratio) / 2 * len(data))
    if lim == 0:
        return data
    return data[lim:-lim]

In [ ]:
threshold = 3.2
sampling_rate = 200e6
freq_threshold = 10e6 / (sampling_rate / 2)
inv_keep_ratio = 0.7
frequency = 1.1e6

In [ ]:
Font_Size = 24
Tick_Size = 24
Fig_Height = 400
Fig_Width = 800

In [ ]:
experiments = [ "280_1", "80_1"]
exp_names = ["D2", "D1"]
fig = go.Figure()

for exp in experiments:
    exp_data = results.get_data(exp)
    centers_sorted = exp_data['centers_sorted']
    labels_sorted = exp_data['labels_sorted']
    tr_obj = exp_data['tr_obj']
    data_preprocessed = tr_obj.data
    data_trim = exp_data['data_trim']
    std_original = np.std(data_trim)

    # Add number to plot_centres to plot reconstructions of all nodes belonging to the cluster
    # plot_centres = [i for i in range(5)]
    plot_centres = np.arange(len(centers_sorted)) 
    plot_centres = plot_centres[centers_sorted['kurtosis']>threshold]
    reconstructions = [np.zeros(len(data_preprocessed)) for _ in range(2)]

    for ii, (node, assignment) in enumerate(
        zip(tr_obj.get_leaf_nodes(freq_threshold=freq_threshold), labels_sorted)
    ):
        if int(assignment) in plot_centres:
            # List reconstructions
            rec_signal = inv.inv_node(node)

            j = 0 if sf.kurtosis(trim(rec_signal, inv_keep_ratio)) > threshold else 1  # shorten name for readability
            reconstructions[j] = reconstructions[j] + rec_signal
        else:
            rec_signal = inv.inv_node(node)
            reconstructions[1] += rec_signal

    reconstructions = [rec * std_original for rec in reconstructions]

    print("Energy of the HIGH-kurtosis broadband [dB]: ",sf.avg_energy(trim(reconstructions[0], inv_keep_ratio)), 
      "\nEnergy of the LOW-kurtosis broadband [dB]:",sf.avg_energy(trim(reconstructions[1], inv_keep_ratio)))
    
    print("Kurtosis of the HIGH-kurtosis broadband: ",sf.kurtosis(trim(reconstructions[0], inv_keep_ratio)), 
      "\nKurtosis of the LOW-kurtosis broadband:",sf.kurtosis(trim(reconstructions[1], inv_keep_ratio)))
    
    print("Skewness of the HIGH-kurtosis broadband: ",sf.skewness(trim(reconstructions[0], inv_keep_ratio)), 
      "\nSkewness of the LOW-kurtosis broadband:",sf.skewness(trim(reconstructions[1], inv_keep_ratio)))

    print("Spectral entropy of the HIGH-kurtosis broadband: ",sf.spectral_entropy(trim(reconstructions[0], inv_keep_ratio), sampling_rate), 
      "\nSpectral entropy of the LOW-kurtosis broadband:",sf.spectral_entropy(trim(reconstructions[1], inv_keep_ratio), sampling_rate))
    fig.add_trace(
        go.Scatter(
            y=reconstructions[0],
            mode="lines",
            name=exp_names[experiments.index(exp)],
        )
    )
    # fig.add_trace(
    #     go.Scatter(
    #         y=reconstructions[1],
    #         mode="lines",
    #         name="lower than threshold",
    #     )
    # )

    # Plot psds
    psd_rec_higher, freq_higher = psd(reconstructions[0], sampling_rate, frequency)
    psd_rec_lower, freq_lower = psd(reconstructions[1], sampling_rate, frequency)

    # Generate polynomial fitting of the PSD curve
    psd_curve_db = ndi.gaussian_filter1d(to_dB(psd_rec_higher), 7)
    psd_curve = 10 ** (psd_curve_db / 10)

    rms = np.sqrt(np.mean(psd_curve**2))
    half_peak = max(psd_curve) / 2
    e_peak = max(psd_curve) / np.e**2

    broadband_lines = {
        "rms": rms, 
        "FWHM": half_peak, 
        "1/e<sup>2</sup> peak": e_peak
    }

    fig2 = go.Figure()

    fig2.add_trace(
        go.Scatter(
            x=freq_higher,
            y=to_dB(psd_rec_higher),
            mode="lines",
            name="higher than threshold",
        )
    )
    fig2.add_trace(
        go.Scatter(
            x=freq_higher,
            y=psd_curve_db,
            name="envelope"
        )
    )

    for line_name, line in broadband_lines.items():
        db_line = 10 * np.log10(line)
        fig2.add_trace(
            go.Scatter(
                x=freq_higher,
                y=db_line*np.ones(len(freq_higher)),
                line=dict(dash="dash"),
                name=line_name
            )
        )

    fig2.update_layout(
        title=f"PSD of reconstructed signals for experiment {exp}",
        width=Fig_Width,
        height=Fig_Height,
        font_size=Font_Size,
        legend=dict(title="", traceorder="normal", font=dict(size=Font_Size)),
        plot_bgcolor="white",
        yaxis=dict(title="PSD [dB]", range=[-100, -35]),
        xaxis=dict(title="f / f<sub>0</sub> [DL]", range=[0, 10]),
    )
    fig2.update_xaxes(
        mirror=True,
        ticks="outside",
        showline=True,
        linecolor="black",
        gridcolor="lightgrey",
    )
    fig2.update_yaxes(
        mirror=True,
        ticks="outside",
        showline=True,
        linecolor="black",
        gridcolor="lightgrey",
    )

    fig2.show()
    if savefigs_flag:
        save_figure(fig2, f"kmeans_broadband_psd_reconstructions_{exp}")

fig.update_layout(
    title=f"Reconstructed signals for experiment",
    width=Fig_Width,
    height=Fig_Height,
    font_size=Font_Size,
    xaxis=dict(title="Samples", tickfont=dict(size=Tick_Size)),
    yaxis=dict(title="Amplitude", tickfont=dict(size=Tick_Size)),
    legend=dict(title="", traceorder="normal", font=dict(size=Font_Size)),
    plot_bgcolor="white",
)
fig.update_xaxes(
    mirror=True,
    ticks="outside",
    showline=True,
    linecolor="black",
    gridcolor="lightgrey",
)
fig.update_yaxes(
    mirror=True,
    ticks="outside",
    showline=True,
    linecolor="black",
    gridcolor="lightgrey",
)
fig.show()

if savefigs_flag:
    save_figure(fig, f"kmeans_broadband_reconstructions_exp_{experiments[0]}_{experiments[1]}")

In [ ]:
print(max(psd_rec_higher), min(psd_rec_higher))